# Restormer Pretrained Pre-Run

This notebook runs the official Restormer real-denoising checkpoint on one FAST micro-CT NetCDF volume. Replace the placeholder paths before running.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESTORMER_ROOT = PROJECT / "external" / "Restormer"
WEIGHTS = RESTORMER_ROOT / "Denoising" / "pretrained_models" / "real_denoising.pth"

# Replace this with one FAST/2min NetCDF volume from your data folder.
INPUT_NC = PROJECT / "data" / "crops_2min" / "sample_01.nc"
OUTPUT_NC = PROJECT / "outputs" / "prerun" / "sample_01_restormer_pretrained.nc"
PREVIEW_PNG = PROJECT / "outputs" / "prerun" / "sample_01_restormer_pretrained.png"
VAR_NAME = "microtom"

PROJECT

Clone the official Restormer repository and download `real_denoising.pth`.

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(PROJECT / "scripts" / "download_restormer_assets.py"),
        "--repo-dir",
        str(RESTORMER_ROOT),
        "--download-real-denoising",
    ],
    check=True,
)

Load the model, compute FAST statistics for the selected input volume, and save the pretrained prediction.

In [ ]:
sys.path.insert(0, str(PROJECT / "src"))

import torch
from microct_denoising.data import compute_fast_mean_std_stats
from microct_denoising.inference import predict_and_save_volume, save_middle_slice_preview
from microct_denoising.models.restormer import build_restormer, load_restormer_weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_restormer(RESTORMER_ROOT).to(device)
load_restormer_weights(model, WEIGHTS, device=device)

stats = compute_fast_mean_std_stats(
    [INPUT_NC],
    var_name=VAR_NAME,
    stats_path=OUTPUT_NC.parent / "single_input_fast_stats.npz",
    force=True,
)

pred_vol = predict_and_save_volume(
    model=model,
    fast_path=INPUT_NC,
    var_name=VAR_NAME,
    stats=stats,
    output_path=OUTPUT_NC,
    model_name="restormer_pretrained_real_denoising",
    device=device,
    channels=3,
    pad_multiple=8,
    tile_size=0,
)
save_middle_slice_preview(pred_vol, PREVIEW_PNG, "Restormer pretrained prediction")

OUTPUT_NC, PREVIEW_PNG